# LLM router that decide which method to use for retrieval

In [1]:
from typing import Callable
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [2]:
from pathlib import Path
import pickle
import os
from pprint import pprint
import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INDEX_DIR = PROJECT_ROOT / "storage" / "faiss_index"
CHUNKS_PATH = PROJECT_ROOT / "storage" / "chunks.pkl"

EMBEDDING_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = FAISS.load_local(
    str(INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Loaded vector store and", len(chunks), "chunks")

Loaded vector store and 205 chunks


In [3]:
# vector retrieval function
def retrieve_vector(query: str, k: int = 5):
    return vector_store.similarity_search(query, k=k)

# MMR retrieval 
def retrieve_mmr(query: str, k: int = 5, fetch_k: int = 20, lambda_mult: float = 0.5):
    return vector_store.max_marginal_relevance_search(
        query,
        k=k,
        fetch_k=fetch_k,
        lambda_mult=lambda_mult,
    )
    
# Metadata filtered retrieval
# pass for now, makes more sense in the agentic RAG setting where we send the filter functions as tools to the agent


# BM25 retrieval
# We have to run this once:

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5 


def retrieve_bm25(query: str, k: int = 5):
    bm25_retriever.k = k
    return bm25_retriever.invoke(query)


# Hybrid retrieval (EnsembleRetriever in LangChain classic)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_for_ensemble = BM25Retriever.from_documents(chunks)
bm25_for_ensemble.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_for_ensemble],
    weights=[0.5, 0.5],  # equal weighting for now
)

def retrieve_hybrid(query: str, k: int = 5):
    docs = hybrid_retriever.invoke(query)
    return docs[:k]

let's use chatgpt to generate some questions for us and then while we are there, ask it to generate a code to compare these retreival methods on a set of questions.


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_MD = OUTPUT_DIR / "retrieval_answer_comparison.md"

CHAT_MODEL = "gpt-5-mini"  # change only if your API account uses a different model id

llm = ChatOpenAI(
    model=CHAT_MODEL,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are answering questions about an academic paper using retrieved context.

Use only the retrieved context.
If the retrieved context is not enough, say that the retrieved chunks do not contain enough evidence.
Do not invent proposition statements, theorem results, or notation.
Answer in a clear academic style.
Keep the answer concise, around 150 to 220 words.
"""
        ),
        (
            "human",
            """Question:
{question}

Retrieved context:
{context}

Answer:
"""
        ),
    ]
)

answer_chain = prompt | llm | StrOutputParser()

In [5]:
retrieval_questions = [
    {
        "id": "Q1",
        "type": "conceptual",
        "question": "What is the central research question of the paper, and what gap in the literature does it address?",
        "why_this_question": "This should test broad conceptual retrieval. Vector and MMR should be useful."
    },
    {
        "id": "Q2",
        "type": "model_setup",
        "question": "How does the paper parameterize the Machine's group-specific false positive and false negative rates using alpha, beta, and delta?",
        "why_this_question": "This tests exact model notation. BM25 and hybrid may help."
    },
    {
        "id": "Q3",
        "type": "formal_result",
        "question": "What does Proposition 1 characterize about the decision maker's optimal action under rational inattention?",
        "why_this_question": "This tests proposition retrieval. BM25 and hybrid should be checked carefully."
    },
    {
        "id": "Q4",
        "type": "fairness_result",
        "question": "How does the fairness landscape differ between the Aware and Unaware decision maker scenarios?",
        "why_this_question": "This tests retrieval across results sections. MMR or hybrid may help."
    },
    {
        "id": "Q5",
        "type": "accuracy_and_audit",
        "question": "Why can aggregate accuracy audits miss distributional harms in the Unaware decision maker scenario?",
        "why_this_question": "This tests implication-oriented retrieval. Vector and MMR should be useful."
    },
]

In [6]:
retrieval_methods = {
    "vector": retrieve_vector,
    "mmr": retrieve_mmr,
    "bm25": retrieve_bm25,
    "hybrid": retrieve_hybrid,
}

In [7]:
results = []
md_lines = []

md_lines.append("# Retrieval Method Answer Comparison")
md_lines.append("")
md_lines.append(f"Model used: `{CHAT_MODEL}`")
md_lines.append("")
md_lines.append("This file compares generated answers across retrieval methods.")
md_lines.append("")
md_lines.append("Retrieval methods compared:")
md_lines.append("")
md_lines.append("- vector")
md_lines.append("- mmr")
md_lines.append("- bm25")
md_lines.append("- hybrid")
md_lines.append("")

total_runs = len(retrieval_questions) * len(retrieval_methods)
run_counter = 0

for q in retrieval_questions:
    question_id = q["id"]
    question_type = q["type"]
    question = q["question"]
    why_this_question = q["why_this_question"]

    md_lines.append("---")
    md_lines.append("")
    md_lines.append(f"## {question_id}: {question}")
    md_lines.append("")
    md_lines.append(f"**Question type:** {question_type}")
    md_lines.append("")
    md_lines.append(f"**Why this question:** {why_this_question}")
    md_lines.append("")

    for method_name, retrieve_fn in retrieval_methods.items():
        run_counter += 1

        print(f"Running {run_counter}/{total_runs}: {question_id} with {method_name}")

        docs = retrieve_fn(question, k=5)

        context_blocks = []

        for rank, doc in enumerate(docs, start=1):
            chunk_id = doc.metadata.get("chunk_id")
            source = doc.metadata.get("source")
            section_title = doc.metadata.get("section_title")
            is_appendix = doc.metadata.get("is_appendix")
            envs = doc.metadata.get("chunk_environments")

            context_blocks.append(
                f"""[Retrieved chunk {rank}]
chunk_id: {chunk_id}
source: {source}
section_title: {section_title}
is_appendix: {is_appendix}
environments: {envs}

{doc.page_content}
"""
            )

        context = "\n\n".join(context_blocks)

        answer = answer_chain.invoke(
            {
                "question": question,
                "context": context,
            }
        )

        retrieved_chunks = []

        for rank, doc in enumerate(docs, start=1):
            retrieved_chunks.append(
                {
                    "rank": rank,
                    "chunk_id": doc.metadata.get("chunk_id"),
                    "source": doc.metadata.get("source"),
                    "section_title": doc.metadata.get("section_title"),
                    "is_appendix": doc.metadata.get("is_appendix"),
                    "environments": doc.metadata.get("chunk_environments"),
                }
            )

        results.append(
            {
                "question_id": question_id,
                "question_type": question_type,
                "question": question,
                "retrieval_method": method_name,
                "answer": answer,
                "retrieved_chunks": retrieved_chunks,
            }
        )

        md_lines.append(f"### Retrieval method: `{method_name}`")
        md_lines.append("")
        md_lines.append("#### Answer")
        md_lines.append("")
        md_lines.append(answer)
        md_lines.append("")
        md_lines.append("#### Retrieved chunks")
        md_lines.append("")

        for item in retrieved_chunks:
            md_lines.append(
                f"- Rank {item['rank']}: "
                f"chunk_id=`{item['chunk_id']}`, "
                f"source=`{item['source']}`, "
                f"section=`{item['section_title']}`, "
                f"appendix=`{item['is_appendix']}`, "
                f"envs=`{item['environments']}`"
            )

        md_lines.append("")

OUTPUT_MD.write_text("\n".join(md_lines), encoding="utf-8")

print("Done.")
print("Saved markdown output to:", OUTPUT_MD)

Running 1/20: Q1 with vector
Running 2/20: Q1 with mmr
Running 3/20: Q1 with bm25
Running 4/20: Q1 with hybrid
Running 5/20: Q2 with vector
Running 6/20: Q2 with mmr
Running 7/20: Q2 with bm25
Running 8/20: Q2 with hybrid
Running 9/20: Q3 with vector
Running 10/20: Q3 with mmr
Running 11/20: Q3 with bm25
Running 12/20: Q3 with hybrid
Running 13/20: Q4 with vector
Running 14/20: Q4 with mmr


KeyboardInterrupt: 